## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
TASK = "regression"
N_TRIALS = 30
INNER_K = 3
K, REPEATS = 5, 3
np.random.seed(SEED)
os.makedirs("weights_V7_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


In [4]:
# Feature interaction hand-off (Feature Interaction section of regression.ipynb):
# Teenhome interacts with most spend features but is not in the 13-feature model set.
EXTRA = ["Teenhome"]
COLS_ALL = REG_FEATURES + EXTRA
X_all = df[COLS_ALL].values.astype(float)

def top_pairs(csv_path, cols, k, must_include=None, exclude=None):
    """Top-k significant (q < 0.05) pairs by effect size where both features are in `cols`."""
    t = pd.read_csv(csv_path)
    t = t[(t.q < 0.05) & t.A.isin(cols) & t.B.isin(cols)]
    if must_include is not None:
        t = t[t.A.eq(must_include) | t.B.eq(must_include)]
    if exclude is not None:
        t = t[~(t.A.eq(exclude) | t.B.eq(exclude))]
    t = t.sort_values("effect", ascending=False)
    return list(zip(t.A.head(k), t.B.head(k)))

PAIRS_13 = top_pairs("results/interaction_reg.csv", REG_FEATURES, 4)
PAIRS_T = top_pairs("results/interaction_reg.csv", COLS_ALL, 5, must_include="Teenhome")
print("pairs (13 features):", PAIRS_13); print("pairs (with Teenhome):", PAIRS_T)
SETS = {
    "13 features":                    dict(cols=REG_FEATURES, pairs=[]),
    "+ Teenhome (14)":                dict(cols=COLS_ALL, pairs=[]),
    "13 features + 4 pair nets":      dict(cols=REG_FEATURES, pairs=PAIRS_13),
    "+ Teenhome + 5 Teenhome pairs":  dict(cols=COLS_ALL, pairs=PAIRS_T),
}

def prep_split(tr_idx, te_idx, set_name):
    cols = SETS[set_name]["cols"]
    ci = [COLS_ALL.index(c) for c in cols]
    sc = StandardScaler().fit(X_all[tr_idx][:, ci])
    return sc.transform(X_all[tr_idx][:, ci]), sc.transform(X_all[te_idx][:, ci]), y_raw[tr_idx], y_raw[te_idx]

idx_all = np.arange(len(y_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
for s_ in SETS:
    a_, b_, _, _ = prep_split(tr_idx, te_idx, s_)
    print(f"{s_:32s} train {a_.shape} test {b_.shape}")

pairs (13 features): [('MntGoldProds', 'NumWebPurchases'), ('MntFruits', 'NumDealsPurchases'), ('MntFishProducts', 'NumWebPurchases'), ('NumWebPurchases', 'NumCatalogPurchases')]
pairs (with Teenhome): [('Teenhome', 'NumCatalogPurchases'), ('Teenhome', 'NumWebVisitsMonth'), ('Teenhome', 'MntWines'), ('Teenhome', 'MntMeatProducts'), ('Teenhome', 'NumStorePurchases')]
13 features                      train (1769, 13) test (443, 13)
+ Teenhome (14)                  train (1769, 14) test (443, 14)
13 features + 4 pair nets        train (1769, 13) test (443, 13)
+ Teenhome + 5 Teenhome pairs    train (1769, 14) test (443, 14)


## Model

In [5]:
# Interaction-depth ladder: the same LightGBM, only allowed to combine 1, 2 or many features per tree.
#   stumps  (1 split per tree)  -> purely additive model, the same family as NAM
#   depth 2 (2 splits per path) -> additive + pairwise interactions, the same family as NA2M
#   full    (V5 run, many leaves) -> any interaction order
import lightgbm as lgb
LGB_CLS = lgb.LGBMClassifier if TASK == "classification" else lgb.LGBMRegressor
def lgb_spec(depth, tier):
    def space(t):
        p = dict(n=t.suggest_int("n", 100, 1000, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                 mcs=t.suggest_int("mcs", 5, 50), sub=t.suggest_float("sub", 0.6, 1.0), col=t.suggest_float("col", 0.5, 1.0),
                 lam=t.suggest_float("lam", 1e-2, 10, log=True))
        if TASK == "classification":
            p["spw"] = t.suggest_categorical("spw", [1.0, POS_W])
        return p
    def make(p, s):
        extra = dict(scale_pos_weight=p["spw"]) if TASK == "classification" else {}
        return LGB_CLS(n_estimators=p["n"], learning_rate=p["lr"], max_depth=depth, num_leaves=2 ** depth,
                       min_child_samples=p["mcs"], subsample=p["sub"], subsample_freq=1, colsample_bytree=p["col"],
                       reg_lambda=p["lam"], n_jobs=1, random_state=s, verbosity=-1, **extra)
    d = dict(tier=tier, space=space, make=make)
    if TASK == "classification":
        d["kind"] = "proba"
    return d
TIER = "curriculum" if TASK == "classification" else "curriculum family"
LADDER = {"LightGBM stumps (additive, depth 1)": lgb_spec(1, TIER),
          "LightGBM depth 2 (pairwise)": lgb_spec(2, TIER)}

In [6]:
def inner_oof(spec, params, X, y, seed):
    """Out-of-fold predictions on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in KFold(INNER_K, shuffle=True, random_state=seed).split(X):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = est.predict(X[b])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) minimising RMSE of the inner out-of-fold predictions."""
    if spec.get("fixed"):
        return {}
    def objective(trial):
        return float(np.sqrt(mean_squared_error(y, inner_oof(spec, spec["space"](trial), X, y, seed))))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    pred = est.predict(Xb)
    mse = mean_squared_error(yb, pred)
    return {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse}, pred

## Sanity Check

In [7]:
a, b = next(KFold(K, shuffle=True, random_state=SEED).split(X_all))
Xa, Xb, ya, yb = prep_split(a, b, "+ Teenhome (14)")
for name, spec in LADDER.items():
    est = spec["make"](dict(n=50, lr=0.1, mcs=20, sub=1.0, col=1.0, lam=1.0, spw=1.0), SEED).fit(Xa, ya)
    leaves = est.booster_.dump_model()["tree_info"][0]["num_leaves"]
    print(f"{name:38s} leaves per tree <= {leaves}")
    assert leaves <= 2 ** est.get_params()["max_depth"]
print("feature set:", "+ Teenhome (14)", Xa.shape)

LightGBM stumps (additive, depth 1)    leaves per tree <= 2
LightGBM depth 2 (pairwise)            leaves per tree <= 4
feature set: + Teenhome (14) (1769, 14)


## Training

In [8]:
cache = "weights_V7_1/_progress_regression_lightgbm.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # same outer folds as V4_1 / V5_1 / V6_1
    for fold, (a, b) in enumerate(KFold(K, shuffle=True, random_state=seed_r).split(X_all), start=1):
        Xa, Xb, ya, yb = prep_split(a, b, "+ Teenhome (14)")
        for base, spec in LADDER.items():
            name = f"{base} | + Teenhome (14)"
            if (name, rep, fold) in done:
                continue
            t0 = time.time()
            params = tune(spec, Xa, ya, seed_r * 100 + fold)
            row, s = fit_eval(spec, params, Xa, ya, Xb, yb, seed_r * 100 + fold)
            state["rows"].append(dict(model=name, tier=spec["tier"], selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      params=params, seconds=round(time.time() - t0, 1)))
            state["scores"].setdefault(f"{name}|r{rep}", np.full(len(y_raw), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done", flush=True)

repeat 1 fold 1 done


repeat 1 fold 2 done


repeat 1 fold 3 done


repeat 1 fold 4 done


repeat 1 fold 5 done


...

repeat 3 fold 1 done


repeat 3 fold 2 done


repeat 3 fold 3 done


repeat 3 fold 4 done


repeat 3 fold 5 done


## K-Fold Summary

In [9]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['r2', 'rmse', 'mae']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
print(kdf.groupby("model")["r2"].agg(["mean", "std"]).round(4).to_string())
json.dump({"k": K, "repeats": REPEATS, "folds": state["rows"], "summary": summary},
          open("weights_V7_1/regression_lightgbm_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V7_1/regression_lightgbm_kfold_scores.npz", **state["scores"])

                                                         mean     std
model                                                                
LightGBM depth 2 (pairwise) | + Teenhome (14)          0.8086  0.0572
LightGBM stumps (additive, depth 1) | + Teenhome (14)  0.7829  0.0501
